In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)
import ipywidgets as widgets

mpl.rcParams["animation.embed_limit"] = 100

print("=" * 60)
print("                  KNN CLASSIFICATION")
print("=" * 60)

print("\nUpload your dataset.")
print("Supported formats: CSV, XLSX, XLS")

try:
    from google.colab import files
    uploaded = files.upload()

    if len(uploaded) == 0:
        raise ValueError("No file was uploaded.")

    filename = list(uploaded.keys())[0]

except ImportError:
    raise RuntimeError(
        "This program is designed for Google Colab. "
        "Please run it in Google Colab."
    )

print("\nUploaded file:", filename)

try:
    if filename.lower().endswith(".csv"):
        try:
            df = pd.read_csv(
                io.BytesIO(uploaded[filename]),
                sep=None,
                engine="python"
            )
        except UnicodeDecodeError:
            df = pd.read_csv(
                io.BytesIO(uploaded[filename]),
                sep=None,
                engine="python",
                encoding="latin1"
            )

    elif filename.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(
            io.BytesIO(uploaded[filename])
        )

    else:
        raise ValueError(
            "Unsupported file format. "
            "Please upload CSV, XLSX or XLS."
        )

except Exception as e:
    print("\nERROR READING DATASET")
    print("---------------------")
    print(e)
    raise

df = df.dropna(
    axis=0,
    how="all"
)

df = df.dropna(
    axis=1,
    how="all"
)

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

if df.empty:
    raise ValueError(
        "The uploaded dataset is empty."
    )

if len(df.columns) < 3:
    raise ValueError(
        "The dataset must contain at least "
        "3 columns: two features and one target."
    )

print("\n" + "=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)

print("\nNumber of Rows    :", df.shape[0])
print("Number of Columns :", df.shape[1])

print("\nColumn Names:")

for i, column in enumerate(df.columns):
    print(
        f"{i + 1}. {column}"
    )

print("\nFirst 5 Rows:")
display(df.head())

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("\n" + "=" * 60)
print("NUMERICAL COLUMNS")
print("=" * 60)

if len(numeric_columns) == 0:
    raise ValueError(
        "No numerical columns were found. "
        "KNN visualization requires numerical features."
    )

for column in numeric_columns:
    print("•", column)

print("\n" + "=" * 60)
print("SELECT TARGET COLUMN")
print("=" * 60)

if "Purchased" in df.columns:
    default_target = "Purchased"
else:
    default_target = df.columns[-1]

target_dropdown = widgets.Dropdown(
    options=list(df.columns),
    value=default_target,
    description="Target:",
    style={
        "description_width": "initial"
    },
    layout=widgets.Layout(
        width="400px"
    )
)

display(target_dropdown)

print("\nSelect the value of K:")

k_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=15,
    step=1,
    description="K:",
    style={
        "description_width": "initial"
    },
    layout=widgets.Layout(
        width="400px"
    )
)

display(k_slider)

run_button = widgets.Button(
    description="Run KNN",
    button_style="success",
    icon="play",
    layout=widgets.Layout(
        width="200px",
        height="40px"
    )
)

display(run_button)

output_area = widgets.Output()
display(output_area)

def run_knn(button):
    with output_area:
        output_area.clear_output(wait=True)

        target_column = target_dropdown.value
        K = k_slider.value

        print("\n")
        print("=" * 60)
        print("                  KNN ANALYSIS")
        print("=" * 60)

        print(
            "\nTarget Column :",
            target_column
        )

        print(
            "K             :",
            K
        )

        numeric_columns = df.select_dtypes(
            include=np.number
        ).columns.tolist()

        feature_columns = [
            column
            for column in numeric_columns
            if column != target_column
        ]

        if len(feature_columns) < 2:
            print("\nERROR")
            print("-" * 60)

            print(
                "At least TWO numerical feature columns "
                "are required."
            )

            print(
                "\nAvailable numerical features:",
                feature_columns
            )

            return

        if len(feature_columns) > 2:
            print("\n" + "=" * 60)
            print("SELECT TWO FEATURES FOR KNN")
            print("=" * 60)

            print(
                "\nYour dataset contains",
                len(feature_columns),
                "numerical features."
            )

            print(
                "For the animated 2-D visualization, "
                "please select exactly TWO."
            )

            feature_selector = widgets.SelectMultiple(
                options=feature_columns,
                description="Features:",
                rows=min(10, len(feature_columns)),
                style={
                    "description_width": "initial"
                },
                layout=widgets.Layout(
                    width="450px"
                )
            )

            display(feature_selector)

            print(
                "\nPlease select exactly TWO features "
                "and click Run KNN again."
            )

            global selected_feature_selector
            selected_feature_selector = feature_selector

            return

        if len(feature_columns) == 2:
            selected_features = feature_columns
        else:
            selected_features = list(
                selected_feature_selector.value
            )

            if len(selected_features) != 2:
                print("\nERROR")
                print("-" * 60)

                print(
                    "Please select exactly TWO "
                    "numerical features."
                )

                return

        feature_columns = selected_features

        print("\nFeatures used:")

        for feature in feature_columns:
            print("•", feature)

        data = df[
            feature_columns + [target_column]
        ].copy()

        before_rows = len(data)
        data = data.dropna()
        removed_rows = before_rows - len(data)

        if removed_rows > 0:
            print(
                "\nMissing-value rows removed:",
                removed_rows
            )

        if len(data) < 10:
            print("\nERROR")
            print("-" * 60)

            print(
                "Dataset contains too few valid "
                "samples after removing missing values."
            )

            return

        X = data[
            feature_columns
        ]

        y = data[
            target_column
        ]

        label_encoder = LabelEncoder()

        y_encoded = label_encoder.fit_transform(
            y.astype(str)
        )

        print("\n" + "=" * 60)
        print("CLASS INFORMATION")
        print("=" * 60)

        print("\nClasses:")

        for i, class_name in enumerate(
            label_encoder.classes_
        ):
            print(
                f"{i} → {class_name}"
            )

        unique_classes, class_counts = np.unique(
            y_encoded,
            return_counts=True
        )

        print("\nClass Distribution:")

        for class_id, count in zip(
            unique_classes,
            class_counts
        ):
            print(
                f"{label_encoder.classes_[class_id]} : {count}"
            )

        if len(unique_classes) < 2:
            print("\nERROR")
            print("-" * 60)

            print(
                "Target column must contain "
                "at least TWO classes."
            )

            return

        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y_encoded,
                test_size=0.20,
                random_state=42,
                stratify=y_encoded
            )

        except ValueError:
            print(
                "\nWarning: Stratified split could not "
                "be performed."
            )

            print(
                "Using a regular random split instead."
            )

            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y_encoded,
                test_size=0.20,
                random_state=42
            )

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(
            X_train
        )

        X_test_scaled = scaler.transform(
            X_test
        )

        if K > len(X_train_scaled):
            print("\nERROR")
            print("-" * 60)

            print(
                f"K = {K} is greater than the "
                f"number of training samples "
                f"({len(X_train_scaled)})."
            )

            print(
                f"\nPlease choose K <= "
                f"{len(X_train_scaled)}."
            )

            return

        model = KNeighborsClassifier(
            n_neighbors=K,
            metric="euclidean"
        )

        model.fit(
            X_train_scaled,
            y_train
        )

        y_pred = model.predict(
            X_test_scaled
        )

        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        print("\n")

        print("=" * 60)
        print("                  RESULTS")
        print("=" * 60)

        print(
            "\nTotal Samples       :",
            len(data)
        )

        print(
            "Training Samples    :",
            len(X_train)
        )

        print(
            "Testing Samples     :",
            len(X_test)
        )

        print(
            "Number of Features  :",
            len(feature_columns)
        )

        print(
            "K                   :",
            K
        )

        print(
            "\nAccuracy            :",
            f"{accuracy * 100:.2f}%"
        )

        all_labels = np.arange(
            len(label_encoder.classes_)
        )

        cm = confusion_matrix(
            y_test,
            y_pred,
            labels=all_labels
        )

        print("\nConfusion Matrix:")
        print(cm)

        print("\nClassification Report:")

        print(
            classification_report(
                y_test,
                y_pred,
                labels=all_labels,
                target_names=[
                    str(x)
                    for x in label_encoder.classes_
                ],
                zero_division=0
            )
        )

        print("\n")

        print("=" * 60)
        print("           STARTING KNN ANIMATION")
        print("=" * 60)

        X_visual = X_train_scaled
        X_test_visual = X_test_scaled

        test_point = X_test_visual[0]
        actual_class = y_test[0]

        predicted_class = model.predict(
            test_point.reshape(
                1,
                -1
            )
        )[0]

        distances = np.sqrt(
            np.sum(
                (
                    X_visual -
                    test_point
                ) ** 2,
                axis=1
            )
        )

        sorted_indices = np.argsort(
            distances
        )

        nearest_indices = sorted_indices[:K]

        max_distance_frames = 30

        total_training_points = len(
            X_visual
        )

        if total_training_points <= max_distance_frames:
            animation_indices = np.arange(
                total_training_points
            )
        else:
            animation_indices = np.linspace(
                0,
                total_training_points - 1,
                max_distance_frames,
                dtype=int
            )

        fig, ax = plt.subplots(
            figsize=(9, 6)
        )

        classes = np.unique(
            y_train
        )

        for class_value in classes:
            points = X_visual[
                y_train == class_value
            ]

            ax.scatter(
                points[:, 0],
                points[:, 1],
                s=70,
                label=(
                    "Class "
                    +
                    str(
                        label_encoder.classes_[
                            class_value
                        ]
                    )
                )
            )

        ax.scatter(
            test_point[0],
            test_point[1],
            marker="*",
            s=350,
            edgecolors="black",
            linewidth=2,
            label="Test Point"
        )

        ax.set_xlabel(
            feature_columns[0],
            fontsize=12
        )

        ax.set_ylabel(
            feature_columns[1],
            fontsize=12
        )

        ax.set_title(
            "KNN Classification - Animated",
            fontsize=16
        )

        ax.grid(
            True,
            alpha=0.3
        )

        ax.legend()

        info = ax.text(
            0.02,
            0.97,
            "",
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=10,
            bbox=dict(
                boxstyle="round",
                facecolor="white",
                alpha=0.9
            )
        )

        lines = []

        def update(frame):
            for line in lines:
                line.remove()

            lines.clear()

            if frame == 0:
                info.set_text(
                    "STEP 1\n\n"
                    "Test point selected\n"
                    "Waiting to calculate distances..."
                )

            elif 1 <= frame <= len(
                animation_indices
            ):
                index = animation_indices[
                    frame - 1
                ]

                point = X_visual[
                    index
                ]

                line, = ax.plot(
                    [
                        test_point[0],
                        point[0]
                    ],
                    [
                        test_point[1],
                        point[1]
                    ],
                    linestyle="--",
                    linewidth=1
                )

                lines.append(line)

                info.set_text(
                    "STEP 2: DISTANCE CALCULATION\n\n"
                    f"Training Point : {index + 1}\n"
                    f"Distance       : "
                    f"{distances[index]:.3f}"
                )

            elif frame == len(
                animation_indices
            ) + 1:
                for index in nearest_indices:
                    point = X_visual[
                        index
                    ]

                    line, = ax.plot(
                        [
                            test_point[0],
                            point[0]
                        ],
                        [
                            test_point[1],
                            point[1]
                        ],
                        linewidth=3
                    )

                    lines.append(line)

                neighbour_classes = y_train[
                    nearest_indices
                ]

                neighbour_names = [
                    label_encoder.classes_[c]
                    for c in neighbour_classes
                ]

                info.set_text(
                    "STEP 3: K NEAREST NEIGHBOURS\n\n"
                    f"K = {K}\n"
                    f"Nearest Points = "
                    f"{nearest_indices + 1}\n"
                    f"Classes = "
                    f"{neighbour_names}"
                )

            elif frame == len(
                animation_indices
            ) + 2:
                for index in nearest_indices:
                    point = X_visual[
                        index
                    ]

                    line, = ax.plot(
                        [
                            test_point[0],
                            point[0]
                        ],
                        [
                            test_point[1],
                            point[1]
                        ],
                        linewidth=3
                    )

                    lines.append(line)

                neighbour_classes = y_train[
                    nearest_indices
                ]

                unique, counts = np.unique(
                    neighbour_classes,
                    return_counts=True
                )

                voting_text = ""

                for c, count in zip(
                    unique,
                    counts
                ):
                    voting_text += (
                        f"Class "
                        f"{label_encoder.classes_[c]}"
                        f" → {count} vote(s)\n"
                    )

                info.set_text(
                    "STEP 4: MAJORITY VOTING\n\n"
                    +
                    voting_text
                )

            else:
                for index in nearest_indices:
                    point = X_visual[
                        index
                    ]

                    line, = ax.plot(
                        [
                            test_point[0],
                            point[0]
                        ],
                        [
                            test_point[1],
                            point[1]
                        ],
                        linewidth=3
                    )

                    lines.append(line)

                info.set_text(
                    "STEP 5: FINAL PREDICTION\n\n"
                    f"K = {K}\n"
                    f"Actual Class    : "
                    f"{label_encoder.classes_[actual_class]}\n"
                    f"Predicted Class : "
                    f"{label_encoder.classes_[predicted_class]}"
                )

            return lines + [info]

        total_frames = (
            len(animation_indices)
            + 4
        )

        anim = FuncAnimation(
            fig,
            update,
            frames=total_frames,
            interval=350,
            repeat=True,
            blit=False
        )

        plt.close(fig)

        animation_html = anim.to_jshtml()

        display(
            HTML(
                animation_html
            )
        )

run_button.on_click(
    run_knn
)

print("\nReady!")

print(
    "Select the target column, choose K, "
    "and click 'Run KNN'."
)

                  KNN CLASSIFICATION

Upload your dataset.
Supported formats: CSV, XLSX, XLS
